# 99_freeze — 사전 고정 기록 (담당: core, Test 열기 전 1회)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Walnutbrown/kdt14th-lendingclub/blob/main/notebooks/99_freeze.ipynb)

Test를 열기 전에 실행합니다. 코드·설정·계약·Dev 데이터·세 모형 산출물(`final_model.joblib`, `models_b/*` 포함)·선택 기록(`selection.json`)·민감도 A(`core16`)의 **일관성과 sha256을 일괄 검증·기록**합니다.

1. `00_config`가 `STAGE = "final"`이고 미결 0개인지 확인합니다(`DRY_RUN = False`).
2. ①②③의 `summary.json`, `bootstrap_results.csv`, `bootstrap_curves.csv`, `final_model.joblib`, `models_b/model_b*.joblib`, `split_hash`, `data_files`, `summary.files`, 모형별 범위 해시·하이퍼파라미터, `outputs/collect/selection.json`, `contract/signs.json`을 한 번에 검증합니다.
3. 검증을 통과한 경우에만 `freeze/freeze_<날짜시각>/`에 코드·계약·요약·선택 기록 사본을 저장하고, 대형 바이너리(`final_model.joblib`, `models_b/*`, `core16/*`, `dev_*.parquet`)를 포함한 **모든 파일 sha256과 노트북 코드 해시**를 `FREEZE_RECORD.json`에 씁니다.
4. 이후 `00b_test_prep`과 `05_test`는 Test를 열기 전에 `verify_freeze()`로 이 `FREEZE_RECORD.json`과 현재 디스크 파일이 완전히 일치하는지 검증합니다.

`DRY_RUN = True`면 검증 결과 표만 출력하고 `freeze/_dryrun_…`에 시험 기록만 남깁니다(`LATEST_FREEZE.txt`는 갱신하지 않음).

In [ ]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
import io, json, hashlib, shutil, time

DRY_RUN = False
if not DRY_RUN:
    assert STAGE == "final" and not PENDING, f"STAGE='final'이고 미결 0개여야 합니다. 지금: STAGE={STAGE}, 미결={list(PENDING)}"
stamp = time.strftime("%Y%m%d_%H%M%S")
FREEZE = FREEZE_DIR / (f"_dryrun_{stamp}" if DRY_RUN else f"freeze_{stamp}")
print("목표 기록 폴더:", FREEZE.relative_to(ROOT), "| DRY_RUN =", DRY_RUN)

In [ ]:
# 1. 전 단계 무결성·버전·혼입 일괄 검증 (단일 읽기 바이트 기준)
NOW_SHARED = config_fingerprint("shared")
NOW_FPS = config_fingerprints()
REQ_SUMMARY = ("model", "a_star", "tau_star_median", "columns", "config_hash", "lib_prep_hash")
REQ_RES = ["b", "a_star", "tau_star", "sr_max", "auc", "brier"]
REQ_CURVE = ["b", "rate", "sr", "tau"]
DEV_FILES = ("dev_X_raw.parquet", "dev_meta.parquet")
is_sha = lambda v: isinstance(v, str) and len(v) == 64 and all(ch in "0123456789abcdef" for ch in v)

def read_hashed(path):
    b = Path(path).read_bytes()
    return b, hashlib.sha256(b).hexdigest()

failures, warnings_log = [], []
artifact_hashes = {}   # ROOT 기준 상대경로 -> 단일 읽기 시점 sha256 (3번 칸 저장 직전 재대조)

# (A) Dev 데이터·합성 데이터 플래그·계약 파일 검증
if (DATA_DIR / "SYNTHETIC_DATA.txt").exists():
    failures.append("data/SYNTHETIC_DATA.txt 존재(합성 데이터로는 본 고정 불가)")

live_dev_sha = {}
for n in DEV_FILES:
    p = DATA_DIR / n
    if not p.exists():
        failures.append(f"data/{n} 없음")
    else:
        live_dev_sha[n] = sha256_file(p)
        artifact_hashes[f"data/{n}"] = live_dev_sha[n]
n_dev_rows = len(pd.read_parquet(DATA_DIR / "dev_meta.parquet")) if (DATA_DIR / "dev_meta.parquet").exists() else None
if (DATA_DIR / "prep_meta.json").exists():
    pm = json.loads((DATA_DIR / "prep_meta.json").read_text(encoding="utf-8"))
    if pm.get("data_files") != live_dev_sha:
        failures.append("data/prep_meta.json의 data_files가 현재 dev_*.parquet과 불일치")
    if pm.get("lib_prep_hash") != LIB_PREP_HASH:
        failures.append(f"data/prep_meta.json lib_prep_hash 불일치({pm.get('lib_prep_hash')} ≠ {LIB_PREP_HASH})")
    if pm.get("prep_hash") != NOW_FPS["prep"]:
        failures.append(f"data/prep_meta.json prep_hash 불일치({pm.get('prep_hash')} ≠ {NOW_FPS['prep']} — SAMPLE/PREP 변경 후 01_prep 재실행 필요)")

cols_contract = {}
for k, fn in (("logit", "columns_logit.json"), ("xgb", "columns_xgb.json")):
    p = CONTRACT_DIR / fn
    if not p.exists():
        failures.append(f"contract/{fn} 없음")
    else:
        cols_contract[k] = json.loads(p.read_text(encoding="utf-8"))
if "xgb" in cols_contract:
    cols_contract["xgb_mono"] = cols_contract["xgb"]
if "logit" in cols_contract and "xgb" in cols_contract and cols_contract["logit"] != cols_contract["xgb"]:
    failures.append("contract/columns_logit.json ≠ columns_xgb.json")
if (CONTRACT_DIR / "signs.json").exists():
    signs_file = json.loads((CONTRACT_DIR / "signs.json").read_text(encoding="utf-8"))
    spec_signs = {out: s for _, (out, _, _, s, _) in FEATURE_SPEC.items()}
    drift = [k for k, v in signs_file.items() if v != spec_signs.get(k, 0)]
    if drift:
        failures.append(f"contract/signs.json과 lib_prep.FEATURE_SPEC 부호 불일치: {drift[:5]}")

# (B) 세 모형(logit, xgb, xgb_mono) 완전 검증
expected_hp = {
    "logit": _json_safe(LOGIT_PARAMS),
    "xgb": _json_safe({**XGB_FIXED, **XGB_PARAMS}),
    "xgb_mono": _json_safe({**XGB_FIXED, **XGB_PARAMS}),
}
outs, splits_seen, data_seen = {}, {}, {}
for m in ("logit", "xgb", "xgb_mono"):
    d = OUT_DIR / m
    m_bad = []
    req_files = ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib")
    missing_f = [n for n in req_files if not (d / n).exists()]
    if missing_f:
        m_bad.append("파일 없음: " + ", ".join(missing_f))
        outs[m] = dict(exists=False, ok=False, issues=m_bad)
        failures.append(f"{m}: {m_bad[0]}")
        continue

    raw = {n: read_hashed(d / n) for n in req_files}
    for n, (_, h) in raw.items():
        artifact_hashes[f"outputs/{m}/{n}"] = h
    s = json.loads(raw["summary.json"][0].decode("utf-8"))
    res = pd.read_csv(io.BytesIO(raw["bootstrap_results.csv"][0]))
    cur = pd.read_csv(io.BytesIO(raw["bootstrap_curves.csv"][0]))

    # models_b 정확히 B개(model_b000..model_b{B-1}) 확인 및 전체 해시 수집
    mb_dir = d / "models_b"
    mb_paths = sorted(mb_dir.glob("model_b*.joblib")) if mb_dir.is_dir() else []
    expected_mb = [f"model_b{i:03d}.joblib" for i in range(B)]
    if [p.name for p in mb_paths] != expected_mb:
        m_bad.append(f"models_b 파일 불일치({len(mb_paths)}개, 기대 {B}개)")
    mb_hashes = {f"models_b/{p.name}": sha256_file(p) for p in mb_paths}
    for rel_mb, h in mb_hashes.items():
        artifact_hashes[f"outputs/{m}/{rel_mb}"] = h

    if not all(k in s for k in REQ_SUMMARY):
        m_bad.append("summary 공통 키 누락: " + str([k for k in REQ_SUMMARY if k not in s]))
    if s.get("model") != m:
        m_bad.append(f"summary.model={s.get('model')!r} ≠ {m!r}")
    if s.get("config_hash") != NOW_SHARED:
        m_bad.append(f"config_hash 불일치({s.get('config_hash')} ≠ {NOW_SHARED})")
    if s.get("lib_prep_hash") != LIB_PREP_HASH:
        m_bad.append(f"lib_prep_hash 불일치({s.get('lib_prep_hash')} ≠ {LIB_PREP_HASH})")

    # 모형별 범위 해시 + 하이퍼파라미터 대조 (②와 ③의 파라미터 혼입·구버전 튜닝 차단)
    s_fps = (s.get("meta") or {}).get("fingerprints") or {}
    if m in s_fps and s_fps[m] != NOW_FPS[m]:
        m_bad.append(f"scope_hash[{m}] 불일치({s_fps[m]} ≠ {NOW_FPS[m]})")
    got_hp = _json_safe(s.get("hyperparameters") or {})
    if m == "logit" and isinstance(got_hp, dict):
        got_hp = {k: v for k, v in got_hp.items() if k != "random_state"}
    if got_hp != expected_hp[m]:
        m_bad.append(f"hyperparameters가 현재 00_config와 다름")

    if m in cols_contract and s.get("columns") != cols_contract[m]:
        m_bad.append("columns ≠ contract")
    if list(res.columns) != REQ_RES:
        m_bad.append("bootstrap_results 열 형식 불일치")
    if list(cur.columns) != REQ_CURVE:
        m_bad.append("bootstrap_curves 열 형식 불일치")
    if s.get("B") != B or len(res) != B or cur["b"].nunique() != B:
        m_bad.append(f"회차 수 불일치(summary B={s.get('B')}, res={len(res)}, curves={cur['b'].nunique()}, 기대={B})")
    if not np.isclose(s.get("a_star", np.nan), res["a_star"].median()):
        m_bad.append("a_star ≠ bootstrap_results 중앙값")

    # 분할 지문·Dev 데이터 지문·결과 파일 지문(PR #3 계약)
    sh = s.get("split_hash")
    if not (isinstance(sh, str) and len(sh) == 12):
        m_bad.append("split_hash 누락/형식 오류(PR #3 반영 후 재실행 필요)")
    else:
        splits_seen[m] = sh
    if s.get("n_samples") is not None and n_dev_rows is not None and int(s["n_samples"]) != n_dev_rows:
        m_bad.append(f"n_samples={s['n_samples']} ≠ 현재 Dev {n_dev_rows}")

    dfp = s.get("data_files")
    if not (isinstance(dfp, dict) and all(is_sha(dfp.get(n)) for n in DEV_FILES)):
        m_bad.append("data_files(Dev parquet sha256) 누락")
    else:
        data_seen[m] = {n: dfp[n] for n in DEV_FILES}
        if live_dev_sha and data_seen[m] != live_dev_sha:
            m_bad.append("data_files가 현재 data/dev_*.parquet과 불일치")

    sf = s.get("files")
    if not (isinstance(sf, dict) and all(sf.get(k) == raw[k][1] for k in ("bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib"))):
        m_bad.append("summary.files가 현재 CSV·final_model.joblib과 불일치/누락(실행 혼입 의심)")
    # models_b 개별 해시가 summary에 있으면 그것도 대조
    s_mb = s.get("models_b_files") or {k: v for k, v in (sf or {}).items() if k.startswith("models_b/")}
    if s_mb:
        if s_mb != mb_hashes:
            m_bad.append("summary의 models_b 해시가 디스크와 불일치(실행 혼입 의심)")
    else:
        warnings_log.append(f"{m}: summary.json에 models_b 개별 해시 없음(고정 시점 스냅샷에는 전체 기록됨)")

    ok = not m_bad
    outs[m] = dict(exists=True, ok=ok, a_star=s.get("a_star"), B=s.get("B"),
                   split_hash=sh, config_hash=s.get("config_hash"), lib_prep_hash=s.get("lib_prep_hash"), issues=m_bad)
    if not ok:
        failures.append(f"{m}: " + "; ".join(m_bad))

if len(splits_seen) == 3 and len(set(splits_seen.values())) != 1:
    failures.append(f"세 모형 split_hash 불일치: {splits_seen}")
if len(data_seen) == 3 and len({tuple(v[n] for n in DEV_FILES) for v in data_seen.values()}) != 1:
    failures.append("세 모형 data_files(Dev parquet) 불일치")

# (C) 모델 선택 기록(outputs/collect/selection.json) 검증: 있으면 엄격 검증, 아직 없으면 참고 경고만 남기고 고정 진행
SEL_PATH = OUT_DIR / "collect" / "selection.json"
sel_summary = None
if not SEL_PATH.exists():
    warnings_log.append("outputs/collect/selection.json 없음(선택 기록을 쓰려면 06_collect 반영 후 다시 고정)")
else:
    sel_bytes, sel_sha = read_hashed(SEL_PATH)
    artifact_hashes["outputs/collect/selection.json"] = sel_sha
    sel = json.loads(sel_bytes.decode("utf-8"))
    s_bad = []
    if sel.get("valid") is not True:
        s_bad.append(f"valid={sel.get('valid')} ({sel.get('reason')})")
    if sel.get("selected_model") not in ("xgb", "xgb_mono"):
        s_bad.append(f"selected_model={sel.get('selected_model')!r}")
    if sel.get("config_hash") != NOW_SHARED:
        s_bad.append(f"config_hash 불일치({sel.get('config_hash')} ≠ {NOW_SHARED})")
    if sel.get("lib_prep_hash") != LIB_PREP_HASH:
        s_bad.append(f"lib_prep_hash 불일치({sel.get('lib_prep_hash')} ≠ {LIB_PREP_HASH})")
    if sel.get("ref_model") != REF_MODEL or sel.get("delta") != DELTA or sel.get("alpha") != ALPHA or sel.get("min_rate") != MIN_RATE:
        s_bad.append("selection의 ref_model/delta/alpha/min_rate가 현재 00_config와 불일치")
    a_map = sel.get("a_star") if isinstance(sel.get("a_star"), dict) else {}
    a_ref = a_map.get(REF_MODEL)
    if not (isinstance(a_ref, (int, float)) and np.isfinite(a_ref) and MIN_RATE <= float(a_ref) <= 1.0):
        s_bad.append(f"selection.a_star[{REF_MODEL!r}] 누락/범위 오류({a_ref!r})")
    if len(splits_seen) == 3 and sel.get("split_hash") != next(iter(splits_seen.values())):
        s_bad.append("selection.split_hash ≠ 세 모형 split_hash")
    if live_dev_sha and sel.get("data_files") != live_dev_sha:
        s_bad.append("selection.data_files ≠ 현재 data/dev_*.parquet")
    sel_inputs = sel.get("inputs") or {}
    req_sel_in = {f"{m}/{fn}" for m in ("logit", "xgb", "xgb_mono") for fn in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv", "final_model.joblib")}
    if not req_sel_in.issubset(sel_inputs.keys()):
        s_bad.append("selection.inputs 필수 항목 누락: " + ", ".join(sorted(req_sel_in - set(sel_inputs))))
    else:
        for rel_in, h_in in sel_inputs.items():
            full_rel = f"outputs/{rel_in}"
            if artifact_hashes.get(full_rel) != h_in:
                s_bad.append(f"selection.inputs[{rel_in}]이 현재 파일과 불일치")
    if s_bad:
        failures.append("selection.json: " + "; ".join(s_bad))
    sel_summary = dict(valid=sel.get("valid"), selected_model=sel.get("selected_model"),
                       a_star=sel.get("a_star"), ref_model=sel.get("ref_model"), sha256=sel_sha)

# (D) 민감도 A (outputs/core16/) 검증 (파일이 하나라도 있으면 3종 + summary 일괄 검증)
core16_dir = OUT_DIR / "core16"
core16_models = [core16_dir / f"{m}.joblib" for m in ("logit", "xgb", "xgb_mono")]
if any(p.exists() for p in core16_models):
    c_bad = []
    if not all(p.exists() for p in core16_models):
        c_bad.append("core16 모형 일부 누락: " + str([p.name for p in core16_models if not p.exists()]))
    if not (core16_dir / "summary.json").exists():
        c_bad.append("outputs/core16/summary.json 없음")
    else:
        cb, ch = read_hashed(core16_dir / "summary.json")
        artifact_hashes["outputs/core16/summary.json"] = ch
        cs = json.loads(cb.decode("utf-8"))
        if cs.get("dry_run") is not False:
            c_bad.append(f"core16 dry_run={cs.get('dry_run')}")
        if cs.get("config_hash") != NOW_SHARED:
            c_bad.append("core16 config_hash 불일치")
        if cs.get("lib_prep_hash") != LIB_PREP_HASH:
            c_bad.append("core16 lib_prep_hash 불일치")
        core_expected = [out for _, (out, *_r, grp) in FEATURE_SPEC.items() if grp == "core"]
        if cs.get("core_cols") != core_expected:
            c_bad.append("core16 core_cols ≠ FEATURE_SPEC")
        for p in core16_models:
            if p.exists():
                h_j = sha256_file(p)
                artifact_hashes[f"outputs/core16/{p.name}"] = h_j
                if isinstance(cs.get("files"), dict) and p.name in cs["files"] and cs["files"][p.name] != h_j:
                    c_bad.append(f"core16/{p.name} 해시 불일치")
        if isinstance(cs.get("data_files"), dict) and live_dev_sha and cs["data_files"] != live_dev_sha:
            c_bad.append("core16 data_files ≠ 현재 Dev parquet")
    if c_bad:
        failures.append("core16: " + "; ".join(c_bad))

for m, info in outs.items():
    print(f"[{m:<8}] {'PASS' if info['ok'] else 'FAIL'} | B={info.get('B')} a*={info.get('a_star')} split={info.get('split_hash')} {info.get('issues') or ''}")
print("[selection]", sel_summary)
for w in warnings_log:
    print("참고:", w)
if failures:
    print("\n실패 항목:")
    for f_msg in failures:
        print(" -", f_msg)
if not DRY_RUN:
    assert not failures, f"사전 고정 검증 실패({len(failures)}건). 위 항목을 고친 뒤 다시 실행하세요."

In [ ]:
# 2. 복사, 해시 기록(FREEZE_RECORD.json), 저장 직전 재대조 및 자체 검증
FREEZE.mkdir(parents=True, exist_ok=False)
code_sha, raw_nb_sha, files_sha = {}, {}, {}

# 코드·계약 스냅샷 복사
(FREEZE / "notebooks").mkdir()
for p in sorted(NB_DIR.glob("*.ipynb")):
    rel = f"notebooks/{p.name}"
    shutil.copy2(p, FREEZE / rel)
    code_sha[rel] = notebook_code_sha256(FREEZE / rel)
    raw_nb_sha[rel] = sha256_file(FREEZE / rel)

if (ROOT / "colab_setup.py").exists():
    shutil.copy2(ROOT / "colab_setup.py", FREEZE / "colab_setup.py")
    files_sha["colab_setup.py"] = sha256_file(FREEZE / "colab_setup.py")

(FREEZE / "contract").mkdir()
for p in sorted(CONTRACT_DIR.glob("*.json")):
    rel = f"contract/{p.name}"
    shutil.copy2(p, FREEZE / rel)
    files_sha[rel] = sha256_file(FREEZE / rel)

# 경량 산출물(요약·곡선·선택·prep_meta)은 복사 + 대형 바이너리 포함 전체는 sha256 기록
copy_rels = [
    *(f"outputs/{m}/{n}" for m in ("logit", "xgb", "xgb_mono") for n in ("summary.json", "bootstrap_results.csv", "bootstrap_curves.csv")),
    "outputs/collect/selection.json",
    "outputs/core16/summary.json",
    "data/prep_meta.json",
]
for rel in copy_rels:
    src = ROOT / rel
    if src.exists():
        dst = FREEZE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        files_sha[rel] = sha256_file(dst)

# 1번 칸에서 검증한 모든 아티팩트(final_model.joblib, models_b/*, core16/*, dev_*.parquet 포함)를 저장 직전 재대조
drifted = []
for rel, h_checked in sorted(artifact_hashes.items()):
    live_h = sha256_file(ROOT / rel)
    if live_h != h_checked:
        drifted.append(rel)
    if rel in files_sha and files_sha[rel] != h_checked:
        drifted.append(rel)
    files_sha[rel] = h_checked
if drifted:
    shutil.rmtree(FREEZE, ignore_errors=True)
    raise RuntimeError(f"사전 고정 도중 파일이 바뀜(중단): {sorted(set(drifted))}")

record = dict(
    stamp=stamp,
    dry_run=DRY_RUN,
    stage=STAGE,
    pending=sorted(PENDING),
    config_hash=NOW_SHARED,
    fingerprints=NOW_FPS,
    lib_prep_hash=LIB_PREP_HASH,
    split_hash=next(iter(splits_seen.values())) if len(set(splits_seen.values())) == 1 and splits_seen else None,
    data_files=live_dev_sha,
    selection=sel_summary,
    config=config_dict(),
    outputs=outs,
    failures=failures,
    warnings=warnings_log,
    code_sha256=code_sha,
    raw_notebook_sha256=raw_nb_sha,
    sha256=files_sha,
    meta=run_meta(),
)
rec_path = FREEZE / "FREEZE_RECORD.json"
rec_path.write_text(json.dumps(record, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")

if not DRY_RUN:
    (FREEZE_DIR / "LATEST_FREEZE.txt").write_text(FREEZE.name + "\n", encoding="utf-8")
    verify_freeze(FREEZE, require_final=True)   # 방금 쓴 고정본 자체 검증
    print("[PASS] verify_freeze 자체 검증 통과")

print(f"기록 완료: 코드 {len(code_sha)}개, 파일 sha256 {len(files_sha)}개 → {rec_path.relative_to(ROOT)}")
print("공지할 값: shared", NOW_SHARED, "| all", NOW_FPS["all"], "| lib_prep", LIB_PREP_HASH, "| freeze_record", sha256_file(rec_path)[:12])